<a href="https://colab.research.google.com/github/RodolfoFerro/beepy5/blob/main/notebooks/Notebook_2_Clasificaci%C3%B3n_con_CNNs_e_Interpretabilidad_con_GradCAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Taller] **Imágenes como datos:** redes neuronales y visión por computadora 🔬

> **Descripción:** En este taller abordaremos el procesamiento de imágenes como un problema de aprendizaje automático, explorando cómo las redes neuronales artificiales procesan y comprenden imágenes, extrayendo patrones y características que escapan a la percepción humana. Usando Python, implementaremos un pipeline completo (desde la imagen cruda hasta los resultados) y visualizaremos qué aprenden los modelos, convirtiendo la "caja negra" en algo comprensible e interpretable. <br>
> **Autor:** Rodolfo Ferro <br>
> **Contacto:** ferro@cimat.mx / [@rodo_ferro](https://www.instagram.com/rodo_ferro/)


## **Notebook 2** - Clasificación con CNNs e Interpretabilidad con GradCAM

---

**Duración estimada:** ~35 minutos  
**Prerequisito:** Imagen como dato (haber ejecutado el Notebook 1).

---

## **El problema:** Clasificación automática de células sanguíneas

Un **hemograma** (conteo diferencial de leucocitos) es uno de los análisis clínicos más solicitados. En un laboratorio tradicional, un hematólogo examina manualmente un frotis de sangre periférica bajo el microscopio e identifica y cuenta cada tipo celular.

Este proceso es:
- **Lento**: entre 15 y 30 minutos por muestra
- **Subjetivo**: depende de la experiencia del observador
- **Costoso**: requiere personal especializado

Un sistema de visión por computadora puede automatizar esta tarea con alta precisión, funcionando de manera consistente las 24 horas.

### **Dataset:** BloodMNIST

BloodMNIST es parte de la colección [MedMNIST](https://medmnist.com/), un benchmark estandarizado de imágenes médicas diseñado para investigación en deep learning biomédico.

| Característica | Valor |
|---------------|-------|
| Fuente original | Dataset de Acevedo et al. (2020), Barcelona |
| Imágenes totales | 17,092 (train: 11,959 / val: 1,712 / test: 3,421) |
| Resolución | 28 × 28 píxeles, RGB |
| Clases | 8 tipos de células sanguíneas |
| Tinción | Giemsa (estándar en hematología) |
| Tarea | Clasificación multiclase |

### Las 8 clases

| ID | Célula | Característica morfológica clave |
|----|--------|----------------------------------|
| 0 | Basófilo | Gránulos morado-negro que oscurecen el núcleo |
| 1 | Eosinófilo | Gránulos rojizos grandes, núcleo bilobulado |
| 2 | Eritroblasto | Núcleo redondo y muy oscuro, citoplasma azulado |
| 3 | Linfocito IG | Linfocito granular grande, citoplasma más abundante |
| 4 | Linfocito | Núcleo redondo oscuro, poco citoplasma |
| 5 | Monocito | El leucocito más grande, núcleo arriñonado o en herradura |
| 6 | Neutrófilo | Núcleo multilobulado (3-5 lóbulos), el más abundante en sangre |
| 7 | Plaqueta | Fragmento celular, sin núcleo, muy pequeño |

### Pregunta central de este notebook

> Una red neuronal puede aprender a clasificar estas células con >90% de precisión.  
> Pero, **¿qué regiones de la imagen está mirando para tomar esa decisión?**

Esa pregunta la responderemos con **GradCAM**.

---

In [ ]:
# Instalación de dependencias

# Ejecutar solo una vez al inicio de la sesión de Colab
!pip install -q medmnist timm scikit-image
print("✅ Dependencias listas")

In [ ]:
# Imports globales

# Objetos numéricos y gráficos
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image

# Modelos / Redes neuronales
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import timm

# Dataset a utilizar
from medmnist import BloodMNIST

# Estilo de gráficas (p/plt)
plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor": "#0d1117",
    "axes.edgecolor": "#444",
    "axes.labelcolor": "#aaa",
    "xtick.color": "#aaa",
    "ytick.color": "#aaa",
    "text.color": "white",
    "image.cmap": "viridis",
})

NOMBRES_CLASE = [
    "Basófilo",
    "Eosinófilo",
    "Eritroblasto",
    "Linfocito (IG)",
    "Linfocito",
    "Monocito",
    "Neutrófilo",
    "Plaqueta"
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Dispositivo: {DEVICE}")
print(f"PyTorch    : {torch.__version__}")

In [ ]:
# Descarga de BloodMNIST
dataset_train = BloodMNIST(split="train", download=True, as_rgb=True)
dataset_test  = BloodMNIST(split="test",  download=True, as_rgb=True)

# Dataset de entrenamiento
imagenes  = np.array([
    np.array(dataset_train[i][0]) for i in range(len(dataset_train))
])
etiquetas = np.array([
    dataset_train[i][1][0]        for i in range(len(dataset_train))
])

# Dataset de pruebas
imagenes_test  = np.array([
    np.array(dataset_test[i][0]) for i in range(len(dataset_test))
])
etiquetas_test = np.array([
    dataset_test[i][1][0]        for i in range(len(dataset_test))
])

print(f"Train: {imagenes.shape}   Test: {imagenes_test.shape}")
print(f"Distribución de clases (train):")
for c in range(8):
    n = (etiquetas == c).sum()
    barra = "█" * (n // 100)
    print(f"  {c} {NOMBRES_CLASE[c]:<16}: {n:5d}  {barra}")

---
## **Sección 1** — El modelo: EfficientNet pre-entrenado

Entrenar una CNN desde cero requeriría miles de imágenes etiquetadas y horas de cómputo. En su lugar, usaremos **transfer learning**:

1. Tomamos **EfficientNet-B0** pre-entrenada en ImageNet (1.2M imágenes, 1000 clases)
2. Reemplazamos solo la capa final por una nueva de 8 salidas (nuestras clases)
3. Las primeras capas ya saben detectar bordes, texturas y formas — las reutilizamos

```
EfficientNet-B0 pre-entrenada
├── Backbone (congelado o ajustable)  ← detecta bordes, texturas, formas
│   ├── MBConv blocks × 16
│   └── Global Average Pool → embedding de 1280 dims
└── Clasificador (nuevo, 8 clases)    ← aprende a distinguir tipos celulares
    └── Linear(1280 → 8)
```

In [ ]:
# Construimos el modelo
modelo = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=8      # Reemplaza la capa final de 1000 por 8 clases
)
modelo = modelo.to(DEVICE)

total_params    = sum(p.numel() for p in modelo.parameters())
trainable_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)

print("EfficientNet-B0 cargado:")
print(f"  Parámetros totales    : {total_params:,}")
print(f"  Parámetros entrenables: {trainable_params:,}")
print()

# Verificamos que la capa final tiene 8 salidas
print("Capa clasificadora:")
print(f"  {modelo.classifier}")   # Debe decir Linear(in=1280, out=8)

In [ ]:
# Pipeline de preprocesamiento

# IMPORTANTE: debe usar exactamente las mismas estadísticas con las que
# fue entrenada la red en ImageNet. Cambiarlas degradaría el rendimiento.
transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],   # media por canal en ImageNet
        std= [0.229, 0.224, 0.225],   # desv. estándar por canal en ImageNet
    )
])

def preparar_batch(imgs_uint8):
    """Convierte lista de arrays uint8 (H,W,3) a tensor (N,3,224,224)."""
    tensores = [transform(Image.fromarray(img)) for img in imgs_uint8]
    return torch.stack(tensores).to(DEVICE)

# Prueba rápida
batch_prueba = preparar_batch([imagenes[0]])
print(f"Shape del tensor de entrada: {batch_prueba.shape}")
print(f"Rango tras normalización   : [{batch_prueba.min():.2f}, {batch_prueba.max():.2f}]")
print()
print("Nota: los valores negativos son normales después de la normalización.")
print("La red espera este rango, no [0, 1] ni [0, 255].")

---
## **Sección 2** — Forward pass y predicciones

El **forward pass** es el proceso de pasar una imagen por todas las capas de la red hasta obtener una predicción. Internamente ocurre en milisegundos y produce:

1. **Logits**: un vector de 8 números (uno por clase), sin restricción de rango
2. **Probabilidades**: aplicando softmax a los logits → suman 1.0
3. **Predicción**: la clase con mayor probabilidad

$$
\hat{p}_k = \frac{e^{z_k}}{\sum_{j=1}^{8} e^{z_j}}, \quad k = 0, \ldots, 7
$$

In [ ]:
# Forward pass en modo inferencia

# Tomamos un ejemplo de cada clase
modelo.eval()  # desactiva dropout y batch norm en modo train

ejemplos = [
    (NOMBRES_CLASE[c], imagenes[np.where(etiquetas == c)[0][0]], c)
    for c in range(8)
]

fig, axes = plt.subplots(2, 4, figsize=(14, 7), dpi=300)

for ax, (nombre, img, clase_real) in zip(axes.flat, ejemplos):
    # Forward pass
    tensor = preparar_batch([img]) # (1, 3, 224, 224)
    with torch.no_grad():
        logits = modelo(tensor) # (1, 8)
    probs = F.softmax(logits, dim=1).squeeze().cpu().numpy() # (8,)
    pred  = probs.argmax()

    # Visualizar imagen
    ax.imshow(img, interpolation="nearest")

    # Color del título según si acertó o no
    color = "#4ade80" if pred == clase_real else "#f87171"
    simbolo = "✓" if pred == clase_real else "✗"
    ax.set_title(
        f"{simbolo} Real: {nombre}\n"
        f"Pred: {NOMBRES_CLASE[pred]} ({probs[pred]*100:.1f}%)",
        fontsize=8, color=color
    )
    ax.axis("off")

fig.suptitle("Predicciones del modelo (sin fine-tuning en BloodMNIST)", fontsize=11)
plt.tight_layout()
plt.show()
print("Verde ✓ = predicción correcta   Rojo ✗ = predicción incorrecta")

In [ ]:
# Distribución de probabilidades para una célula

# Observemos con detalle qué tan seguro está el modelo
clase_analizar = 0 # Cambiamos para ver otra clase

img_an = imagenes[np.where(etiquetas == clase_analizar)[0][0]]

with torch.no_grad():
    logits_an = modelo(preparar_batch([img_an]))
probs_an = F.softmax(logits_an, dim=1).squeeze().cpu().numpy()
pred_an  = probs_an.argmax()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=300,
                          gridspec_kw={"width_ratios": [1, 2.5]})

axes[0].imshow(img_an, interpolation="nearest")
axes[0].set_title(f"Real: {NOMBRES_CLASE[clase_analizar]}", fontsize=10)
axes[0].axis("off")

colores_barras = ["#4ade80" if i == clase_analizar else
                  "#f87171" if i == pred_an else "#555"
                  for i in range(8)]
barras = axes[1].barh(NOMBRES_CLASE, probs_an * 100,
                       color=colores_barras, edgecolor="#333")
axes[1].set_xlabel("Probabilidad (%)")
axes[1].set_xlim(0, 105)
axes[1].set_title("Distribución softmax", fontsize=10)

for barra, prob in zip(barras, probs_an):
    axes[1].text(prob * 100 + 1, barra.get_y() + barra.get_height() / 2,
                 f"{prob*100:.1f}%", va="center", fontsize=8, color="white")

plt.tight_layout()
plt.show()

print(f"Predicción: {NOMBRES_CLASE[pred_an]}  ({probs_an[pred_an]*100:.2f}%)")
print(f"Clase real: {NOMBRES_CLASE[clase_analizar]}")
print(f"¿Correcta?: {"✓ Sí" if pred_an == clase_analizar else "✗ No"}")

---
## **Sección 3** — Extracción de features intermedias

Antes de llegar a la predicción, la red pasa por decenas de capas que transforman la imagen en representaciones cada vez más abstractas. Podemos interceptar estas representaciones usando **hooks** de PyTorch.

Un hook es una función que se ejecuta automáticamente cada vez que pasa información por una capa específica, sin modificar el modelo.

In [ ]:
# Capturar mapas de activación de la última capa convolucional

# En EfficientNet-B0, la última capa convolucional es "blocks[-1]"
# Sus mapas de activación tienen shape (1, 112, 7, 7) antes del pooling

activaciones_guardadas = {}

def hook_activaciones(module, input, output):
    """Se ejecuta en cada forward pass y guarda la salida de la capa."""
    activaciones_guardadas["ultima_conv"] = output

# Registrar el hook en el último bloque convolucional
capa_objetivo = modelo.blocks[-1]
handle_act = capa_objetivo.register_forward_hook(hook_activaciones)

# Forward pass para capturar activaciones
img_demo = imagenes[np.where(etiquetas == 6)[0][0]]  # neutrófilo
tensor_demo = preparar_batch([img_demo])

with torch.no_grad():
    logits_demo = modelo(tensor_demo)

maps_act = activaciones_guardadas["ultima_conv"]  # (1, C, H", W")
print(f"Shape de los mapas de activación: {maps_act.shape}")
print(f"  → 1 imagen, {maps_act.shape[1]} filtros, mapa espacial {maps_act.shape[2]}×{maps_act.shape[3]}")
print()
print("Cada uno de los", maps_act.shape[1], "filtros detecta un patrón distinto.")
print("GradCAM nos dirá cuáles son más importantes para la predicción.")

In [ ]:
# Visualizar algunos mapas de activación
# Cada mapa es una "respuesta" de un filtro particular
maps_np = maps_act.squeeze(0).cpu().numpy()  # (C, H", W")

N_MOSTRAR = 16
fig, axes = plt.subplots(4, 4, figsize=(10, 10), dpi=300)

for i, ax in enumerate(axes.flat):
    mapa = maps_np[i]  # (7, 7)
    ax.imshow(mapa, cmap="gray", interpolation="nearest")
    ax.set_title(f"Filtro {i}", fontsize=7)
    ax.axis("off")

fig.suptitle(
    f"Primeros {N_MOSTRAR} mapas de activación — última capa conv\n"
    f"({NOMBRES_CLASE[6]}, resolución {maps_np.shape[1]}×{maps_np.shape[2]} px)",
    fontsize=10
)
plt.tight_layout()
plt.show()

print("Cada mapa responde a un patrón diferente de la imagen.")
print("Algunos resaltan el núcleo, otros el borde, otros el citoplasma.")
print("GradCAM combina estos mapas pesados por su importancia para la clase predicha.")

# Remover hook (buena práctica)
handle_act.remove()

---
## **Sección 4** — GradCAM desde cero

**Gradient-weighted Class Activation Mapping** (Selvaraju et al., ICCV 2017) responde la pregunta:

> ¿Qué regiones de la imagen influyen más en la predicción de la clase $c$?

### **Algoritmo en 4 pasos:**

**Paso 1 — Forward pass:** obtener los mapas de activación $A^k \in \mathbb{R}^{H" \times W"}$ de la última capa convolucional.

**Paso 2 — Backward pass:** calcular los gradientes de la puntuación de clase $y^c$ respecto a cada mapa:
$$\frac{\partial y^c}{\partial A^k_{ij}}$$

**Paso 3 — Pesos de importancia:** promedio espacial de los gradientes (Global Average Pooling sobre gradientes):
$$\alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A^k_{ij}}$$

**Paso 4 — Mapa de calor:** combinación lineal ponderada + ReLU (solo regiones con efecto positivo sobre la clase):
$$L^c_{\text{GradCAM}} = \text{ReLU}\!\left(\sum_k \alpha_k^c\, A^k\right)$$

El resultado es un mapa de baja resolución (7×7) que se escala al tamaño original y se superpone sobre la imagen.

In [ ]:
# Implementación de GradCAM

class GradCAM:
    """
    Implementación de GradCAM usando hooks de PyTorch.
    No requiere modificar el modelo ni instalar librerías externas.
    """

    def __init__(self, modelo, capa_objetivo):
        self.modelo = modelo
        self._activaciones = None
        self._gradientes   = None

        # Hook forward: captura los mapas de activación
        self._h_act = capa_objetivo.register_forward_hook(
            lambda m, i, o: setattr(self, "_activaciones", o)
        )
        # Hook backward: captura los gradientes de esos mapas
        self._h_grad = capa_objetivo.register_full_backward_hook(
            lambda m, gi, go: setattr(self, "_gradientes", go[0])
        )

    def __call__(self, tensor_imagen, clase=None):
        """
        Calcula el mapa GradCAM para una imagen.

        Args:
            tensor_imagen: tensor (1, 3, 224, 224)
            clase: índice de clase objetivo (None → usa la predicha)

        Returns:
            heatmap: array (224, 224) en [0, 1]
            clase_pred: índice de la clase predicha
            probs: distribución softmax (8,)
        """
        self.modelo.eval()

        # Paso 1: Forward pass
        logits = self.modelo(tensor_imagen)          # (1, 8)
        probs  = F.softmax(logits, dim=1)            # (1, 8)
        clase_pred = logits.argmax(dim=1).item()

        if clase is None:
            clase = clase_pred

        # Paso 2: Backward pass sobre la clase objetivo
        self.modelo.zero_grad()
        logits[0, clase].backward()                  # gradiente de y^c

        # Paso 3: Pesos de importancia α_k
        # Gradientes: (1, C, H", W") → media sobre H" y W" → (C,)
        gradientes  = self._gradientes.squeeze(0)    # (C, H", W")
        alpha       = gradientes.mean(dim=(1, 2))    # (C,)  ← Global Avg Pool

        # Paso 4: Mapa de calor
        activaciones = self._activaciones.squeeze(0) # (C, H", W")
        # Suma ponderada: Σ_k α_k * A^k
        cam = torch.einsum("k,khw->hw", alpha, activaciones)  # (H", W")
        cam = F.relu(cam)                            # ReLU: solo efecto positivo

        # Normalizar a [0, 1] y escalar a 224×224
        cam = cam.detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        heatmap = np.array(
            Image.fromarray((cam * 255).astype(np.uint8)).resize(
                (224, 224), Image.BILINEAR
            )
        ) / 255.0

        return heatmap, clase_pred, probs.detach().cpu().numpy().squeeze()

    def eliminar_hooks(self):
        """Limpiar hooks al terminar (buena práctica)."""
        self._h_act.remove()
        self._h_grad.remove()


# Instanciar GradCAM sobre la última capa conv de EfficientNet-B0
gradcam = GradCAM(modelo, capa_objetivo=modelo.blocks[-1])
print("GradCAM instanciado ✅")
print(f"Capa objetivo: modelo.blocks[-1]  →  {type(modelo.blocks[-1]).__name__}")

In [ ]:
# Función auxiliar de visualización
def visualizar_gradcam(img_uint8, heatmap, clase_real, clase_pred,
                       probs, nombre_real, nombre_pred, ax_row):
    """
    Dibuja en una fila de 4 ejes:
    [imagen original] [mapa de calor] [superposición] [distribución de probs]
    """
    # Imagen original escalada a 224×224 para coincidir con el heatmap
    img224 = np.array(Image.fromarray(img_uint8).resize((224, 224)))

    # Mapa de calor coloreado (jet: azul→rojo)
    heatmap_color = cm.jet(heatmap)[..., :3]  # (224,224,3)

    # Superposición: 50% imagen + 50% heatmap
    img_norm = img224.astype(float) / 255.0
    superposicion = 0.5 * img_norm + 0.5 * heatmap_color
    superposicion = np.clip(superposicion, 0, 1)

    ax_row[0].imshow(img224, interpolation="bilinear")
    ax_row[0].set_title("Original", fontsize=8)
    ax_row[0].axis("off")

    ax_row[1].imshow(heatmap, cmap="jet", vmin=0, vmax=1,
                     interpolation="bilinear")
    ax_row[1].set_title("GradCAM (mapa)", fontsize=8)
    ax_row[1].axis("off")

    ax_row[2].imshow(superposicion, interpolation="bilinear")
    correcto = clase_real == clase_pred
    simbolo  = "✓" if correcto else "✗"
    color_t  = "#4ade80" if correcto else "#f87171"
    ax_row[2].set_title(
        f"{simbolo} Pred: {nombre_pred}\n({probs[clase_pred]*100:.1f}%)",
        fontsize=8, color=color_t
    )
    ax_row[2].axis("off")

    colores_b = ["#4ade80" if i == clase_real else
                 "#f87171" if i == clase_pred else "#444"
                 for i in range(8)]
    ax_row[3].barh(range(8), probs * 100, color=colores_b)
    ax_row[3].set_yticks(range(8))
    ax_row[3].set_yticklabels(
        [n[:8] for n in NOMBRES_CLASE], fontsize=7
    )
    ax_row[3].set_xlabel("%", fontsize=7)
    ax_row[3].set_xlim(0, 105)
    ax_row[3].set_title(f"Real: {nombre_real}", fontsize=8)

print("Función de visualización lista ✅")

In [ ]:
# GradCAM sobre las 8 clases
fig, axes = plt.subplots(8, 4, figsize=(14, 28), dpi=300)

for clase_id in range(8):
    idx = np.where(etiquetas == clase_id)[0][0]
    img = imagenes[idx]
    tensor = preparar_batch([img])

    heatmap, clase_pred, probs = gradcam(tensor)

    visualizar_gradcam(
        img, heatmap,
        clase_real=clase_id,
        clase_pred=clase_pred,
        probs=probs,
        nombre_real=NOMBRES_CLASE[clase_id],
        nombre_pred=NOMBRES_CLASE[clase_pred],
        ax_row=axes[clase_id]
    )
    # Etiqueta de fila
    axes[clase_id, 0].set_ylabel(
        f"{clase_id}: {NOMBRES_CLASE[clase_id]}",
        fontsize=8, color="#aaa", labelpad=5
    )

fig.suptitle(
    "GradCAM — ¿Qué regiones activa la red para cada tipo celular?\n"
    "Rojo = región más importante para la predicción",
    fontsize=12, y=1.005
)
plt.tight_layout()
plt.show()

### 🏋️ **Ejercicio 4** — Explorar GradCAM en distintas instancias

Observa el mapa de calor y responde:
- ¿La región más activada coincide con la estructura morfológica clave de esa clase?
- ¿Qué pasa cuando el modelo se equivoca? ¿En qué región del mapa se equivocó?
- ¿Hay alguna clase donde el mapa sea difuso (sin una región clara)? ¿A qué podría deberse?

In [ ]:
# Ejercicio 4: explorar GradCAM en la célula que elijas
clase_id  = 0   # Cambia (0-7)
instancia = 1   # Cambia para ver otra célula de la misma clase

# También puedes forzar la clase objetivo del gradiente (no necesariamente la predicha)
# None → usa la clase predicha por el modelo
clase_objetivo = None   # Prueba cambiarlo a un número (0-7)

idx_ej4  = np.where(etiquetas == clase_id)[0][instancia]
img_ej4  = imagenes[idx_ej4]
tensor_ej4 = preparar_batch([img_ej4])

heatmap_ej4, pred_ej4, probs_ej4 = gradcam(tensor_ej4, clase=clase_objetivo)

fig, axes = plt.subplots(1, 4, figsize=(14, 4), dpi=300)
visualizar_gradcam(
    img_ej4, heatmap_ej4,
    clase_real  = clase_id,
    clase_pred  = pred_ej4,
    probs       = probs_ej4,
    nombre_real = NOMBRES_CLASE[clase_id],
    nombre_pred = NOMBRES_CLASE[pred_ej4],
    ax_row      = axes
)
clase_obj_str = NOMBRES_CLASE[clase_objetivo] if clase_objetivo is not None else "predicha"
fig.suptitle(
    f"Ejercicio 4 — {NOMBRES_CLASE[clase_id]} (instancia {instancia})"
    f" | Gradiente hacia clase: {clase_obj_str}",
    fontsize=10, y=1.02
)
plt.tight_layout()
plt.show()

---
## **Sección 5** — Embeddings y similitud: cerrando el ciclo

En el Notebook 1 vimos que los embeddings agrupan células del mismo tipo. Aquí lo confirmamos con más células y una visualización más rica.

Además, conectamos este resultado con la motivación original del taller: un sistema de búsqueda por similitud visual (como el detector de cartas) usa exactamente este espacio de embeddings.

In [ ]:
# Extraer embeddings del penúltimo layer

# Usamos el mismo modelo pero con num_classes=0 para obtener el embedding
extractor = timm.create_model("efficientnet_b0", pretrained=True, num_classes=0)
extractor = extractor.to(DEVICE).eval()

def imagen_a_embedding(img_uint8):
    pil    = Image.fromarray(img_uint8)
    tensor = transform(pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        emb = extractor(tensor).squeeze(0).cpu().numpy()
    return emb / np.linalg.norm(emb)

# Extraer N_POR_CLASE embeddings por clase
N_POR_CLASE = 8
embs, labels = [], []

print(f"Extrayendo {N_POR_CLASE} embeddings por clase...")
for c in range(8):
    idxs = np.where(etiquetas == c)[0][:N_POR_CLASE]
    for idx_c in idxs:
        embs.append(imagen_a_embedding(imagenes[idx_c]))
        labels.append(c)
    print(f"  {NOMBRES_CLASE[c]:<16} ✓")

embs   = np.array(embs)    # (64, 1280)
labels = np.array(labels)  # (64,)
print(f"\nShape: {embs.shape}")

**Similitud intra-clase** es el promedio de similitud coseno entre todos los pares de células de la misma clase. Por ejemplo, tomas los 8 embeddings de neutrófilos y calculas qué tan parecidos son entre sí.

**Similitud inter-clase** es el promedio de similitud coseno entre las células de esa clase y todas las células de las demás clases. Para los neutrófilos, qué tan parecidos son sus embeddings a los de linfocitos, monocitos, eritroblastos, etc.
La interpretación es sencilla:

- Si intra > inter, entonces los embeddings de esa clase forman un cluster compacto y bien separado del resto. El modelo "ve" esa célula como algo distinto a las demás. Buena señal.
- Si intra ≈ inter o se invierten, entonces el modelo confunde esa clase con otras en el espacio de representación, aunque pueda acertar por otras razones en la clasificación final.

In [ ]:
# Matriz de similitud coseno con anotaciones
sim_matrix = embs @ embs.T   # (64, 64)

fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=300,
                          gridspec_kw={"width_ratios": [2, 1]})

# Izquierda: matriz de similitud
im = axes[0].imshow(sim_matrix, cmap="RdYlGn", vmin=0.5, vmax=1.0, aspect="auto")
plt.colorbar(im, ax=axes[0], label="Similitud coseno", fraction=0.025)

tick_pos    = [c * N_POR_CLASE + N_POR_CLASE // 2 for c in range(8)]
tick_labels = [n[:7] for n in NOMBRES_CLASE]
axes[0].set_xticks(tick_pos)
axes[0].set_xticklabels(tick_labels, rotation=35, ha="right", fontsize=8)
axes[0].set_yticks(tick_pos)
axes[0].set_yticklabels(tick_labels, fontsize=8)

for i in range(1, 8):
    pos = i * N_POR_CLASE - 0.5
    axes[0].axhline(pos, color="white", lw=1.5)
    axes[0].axvline(pos, color="white", lw=1.5)

axes[0].set_title(
    f"Similitud coseno entre embeddings\n({N_POR_CLASE} células × 8 clases = {N_POR_CLASE*8} embeddings)",
    fontsize=10
)

# Derecha: similitud promedio intra vs inter por clase
sims_intra_c = []
sims_inter_c = []

for c in range(8):
    mask_c = labels == c

    # Intra: promedio de similitud dentro del bloque de la clase
    bloque = sim_matrix[np.ix_(mask_c, mask_c)]
    np.fill_diagonal(bloque, np.nan)
    sims_intra_c.append(np.nanmean(bloque))

    # Inter: promedio de similitud con todas las otras clases
    mask_otras = ~mask_c
    sims_inter_c.append(sim_matrix[np.ix_(mask_c, mask_otras)].mean())

x   = np.arange(8)
ancho = 0.35
axes[1].bar(x - ancho/2, sims_intra_c, ancho,
            label="Intra-clase", color="#4ade80", alpha=0.85)
axes[1].bar(x + ancho/2, sims_inter_c, ancho,
            label="Inter-clase", color="#f87171", alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels([n[:6] for n in NOMBRES_CLASE],
                         rotation=40, ha="right", fontsize=8)
axes[1].set_ylabel("Similitud coseno promedio")
axes[1].set_ylim(0.6, 1.0)
axes[1].legend(fontsize=9)
axes[1].set_title("Separabilidad por clase", fontsize=10)

plt.tight_layout()
plt.show()

print("Interpretación:")
print("  Verde (intra) > Rojo (inter) → la clase tiene un cluster bien definido")
print("  Si se invierten, el modelo confunde esa clase con otras")

In [ ]:
# Búsqueda por similitud visual

# Dado una célula de consulta, encontrar las más similares en el conjunto

CLASE_CONSULTA = 1   # Cambia para consultar otra clase
IDX_CONSULTA   = 0   # Cambia para usar otra instancia
K_VECINOS      = 5   # Cuántos resultados mostrar

idx_consulta = np.where(etiquetas == CLASE_CONSULTA)[0][IDX_CONSULTA]
img_consulta = imagenes[idx_consulta]
emb_consulta = imagen_a_embedding(img_consulta)

# Calcular similitud con todos los embeddings del conjunto
sims_consulta = embs @ emb_consulta   # (64,)
top_k_idx     = sims_consulta.argsort()[::-1][:K_VECINOS + 1]  # +1 porque incluye a sí mismo

fig, axes = plt.subplots(1, K_VECINOS + 2, figsize=(15, 3.5), dpi=300)

# Imagen de consulta
axes[0].imshow(img_consulta, interpolation="nearest")
axes[0].set_title(f"CONSULTA\n{NOMBRES_CLASE[CLASE_CONSULTA]}", fontsize=8, color="yellow")
axes[0].axis("off")

# Separador
axes[1].text(0.5, 0.5, "→\nTop-K\nsimilares", ha="center", va="center",
             fontsize=10, color="#aaa")
axes[1].axis("off")

# Resultados
rank = 1
for idx_res in top_k_idx:
    if rank > K_VECINOS:
        break
    # saltar la propia imagen de consulta
    # (buscamos en el conjunto de embs, que incluye la consulta si fue incluida)
    clase_res = labels[idx_res]
    sim_res   = sims_consulta[idx_res]

    # recuperar la imagen original del dataset usando el label
    img_res = imagenes[np.where(etiquetas == clase_res)[0][idx_res % N_POR_CLASE]]

    correcto = clase_res == CLASE_CONSULTA
    color_t  = "#4ade80" if correcto else "#f87171"
    axes[rank + 1].imshow(img_res, interpolation="nearest")
    axes[rank + 1].set_title(
        f"#{rank} sim={sim_res:.3f}\n{NOMBRES_CLASE[clase_res]}",
        fontsize=8, color=color_t
    )
    axes[rank + 1].axis("off")
    rank += 1

fig.suptitle(
    f"Búsqueda por similitud visual — top-{K_VECINOS} más similares\n"
    "Verde = misma clase · Rojo = clase diferente",
    fontsize=10, y=1.05
)
plt.tight_layout()
plt.show()

print("💡 Esto es exactamente el principio del detector de cartas de Magic:")
print("   embedding(foto) → buscar top-K más cercanos en el índice → resultado")
print("   La única diferencia es la escala (27,000 cartas en lugar de 64 células).")

In [ ]:
# Limpiar hooks al terminar
gradcam.eliminar_hooks()
print("Hooks eliminados ✅")

---
## **Resumen del Notebook 2**

| Concepto | Lo que vimos |
|----------|--------------|
| **BloodMNIST** | 17K imágenes de células sanguíneas, 8 clases, tinción Giemsa |
| **Transfer learning** | EfficientNet-B0 pre-entrenada en ImageNet, capa final adaptada a 8 clases |
| **Forward pass** | Imagen → logits → softmax → predicción con distribución de probabilidad |
| **Hooks de PyTorch** | Interceptar activaciones y gradientes sin modificar el modelo |
| **GradCAM** | 4 pasos: forward → backward → pesos α → mapa de calor con ReLU |
| **Interpretabilidad** | La red mira el núcleo en linfocitos, los gránulos en eosinófilos, el tamaño en monocitos |
| **Búsqueda por similitud** | Embeddings + similitud coseno = sistema de recuperación visual |

### **Reflexión final**

GradCAM nos mostró que el modelo, **sin haber visto nunca células sanguíneas durante su entrenamiento**, activa regiones morfológicamente relevantes: el núcleo multilobulado del neutrófilo, los gránulos del eosinófilo, la forma arriñonada del monocito.

Esto es evidencia de que las representaciones aprendidas en ImageNet capturan características visuales generales — bordes, texturas, formas — que transfieren bien a dominios biomédicos.

Sin embargo, si el mapa de calor no coincide con la morfología esperada, eso es una **señal de alarma**: el modelo puede estar usando un atajo (*shortcut*) en lugar de la estructura biológica real.

---

### Recursos para profundizar

- 📄 [GradCAM — Selvaraju et al., ICCV 2017](https://arxiv.org/abs/1610.02391)
- 📄 [BloodMNIST — Acevedo et al., 2020](https://doi.org/10.3390/diagnostics10090690)
- 📄 [MedMNIST benchmark](https://medmnist.com/)
- 📄 [Feature Visualization — Olah et al., Distill 2017](https://distill.pub/2017/feature-visualization/)
- 📦 [timm — PyTorch Image Models](https://huggingface.co/timm)

---
> Contenido por **Rodolfo Ferro**. Contacto: [ferro@cimat.mx](ferro@cimat.mx) <br>
BeePy 5.0, 2026.